# Decision Tree с нуля

Реализация дерева решений (CART) с критерием Джини и рекурсивным построением.  
Верификация: сравнение accuracy с `sklearn.tree.DecisionTreeClassifier` на датасетах Iris и Breast Cancer.

> На Iris оба дерева дают 1.0 — датасет слишком простой. Breast Cancer показывает реальное сравнение: точное совпадение до последней цифры.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [ ]:
class Node:
    """
    Узел дерева решений.
    """

    def __init__(self, feature_idx=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


def gini(y):
    # Gini = 1 - sum(p_k^2)
    _, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return 1 - (probs ** 2).sum()


def best_split(X, y):
    # Перебор всех признаков и порогов, минимизируем взвешенный джини
    best_feature, best_threshold, best_gini = None, None, np.inf

    for feature_idx in range(X.shape[1]):
        thresholds = np.unique(X[:, feature_idx])

        for threshold in thresholds:
            left_y = y[X[:, feature_idx] >= threshold]
            right_y = y[X[:, feature_idx] < threshold]

            # Взвешенный джини: n_L/n * Gini(L) + n_R/n * Gini(R)
            gini_split = (len(left_y) * gini(left_y) +
                          len(right_y) * gini(right_y)) / len(y)

            if gini_split < best_gini:
                best_gini = gini_split
                best_feature = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold


class DecisionTree:
    """
    Дерево решений для классификации (критерий Джини, CART).
    """

    def __init__(self, max_depth=5, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.root = None

    def fit(self, X, y):
        self.root = self._build(X, y, depth=0)
        return self

    def _build(self, X, y, depth):
        # Условия останова — вернуть листовой узел
        if len(y) < self.min_samples or depth >= self.max_depth or gini(y) == 0:
            return Node(value=np.bincount(y).argmax())

        feature, threshold = best_split(X, y)

        left_mask = X[:, feature] >= threshold
        left = self._build(X[left_mask], y[left_mask], depth + 1)
        right = self._build(X[~left_mask], y[~left_mask], depth + 1)

        return Node(feature, threshold, left, right)

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_idx] >= node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

In [ ]:
# Iris — простой датасет, оба дерева дают 1.0
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_tree = DecisionTree(max_depth=5)
my_tree.fit(X_train, y_train)

sk_tree = DecisionTreeClassifier(max_depth=5)
sk_tree.fit(X_train, y_train)

print(f'Iris — наше дерево:  {accuracy_score(y_test, my_tree.predict(X_test)):.3f}')
print(f'Iris — sklearn:      {accuracy_score(y_test, sk_tree.predict(X_test)):.3f}')

In [ ]:
# Breast Cancer — реальное сравнение
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_tree = DecisionTree(max_depth=5)
my_tree.fit(X_train, y_train)

sk_tree = DecisionTreeClassifier(max_depth=5)
sk_tree.fit(X_train, y_train)

print(f'Breast Cancer — наше дерево:  {accuracy_score(y_test, my_tree.predict(X_test)):.4f}')
print(f'Breast Cancer — sklearn:      {accuracy_score(y_test, sk_tree.predict(X_test)):.4f}')